# AI-Powered Energy Consumption Forecasting
### End-to-End Walkthrough Notebook for Portfolio Review

This notebook demonstrates:
1. Loading hourly energy consumption data (MWh).
2. Leakage-free feature engineering (calendar, autoregressive lags, rolling statistics).
3. Chronological train/test splitting (80% train / 20% test).
4. Benchmarking a **Random Forest Regressor** against a **Naive Persistence Baseline**.
5. Evaluating with MAE, RMSE, and $R^2$.
6. Generating a 24-hour recursive autoregressive forecast.

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path
curr = Path.cwd().resolve()
PROJECT_ROOT = curr if (curr / 'src').exists() else curr.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import load_energy_data, clean_energy_data
from src.feature_engineering import create_feature_pipeline
from src.model import chronological_split, NaivePersistenceBaseline, train_random_forest, calculate_metrics
from src.forecasting import generate_recursive_24h_forecast

print("Modules imported successfully!")

Modules imported successfully!


## 1. Load and Inspect Energy Data

In [1]:
csv_path = PROJECT_ROOT / "data" / "energy.csv"
df_raw, is_synthetic = load_energy_data(csv_path)
df_clean = clean_energy_data(df_raw)

print(f"Total Records: {len(df_clean)}")
print(f"Date Range   : {df_clean['Datetime'].min()} to {df_clean['Datetime'].max()}")
print(f"Synthetic Data Flag: {is_synthetic}")
df_clean.head()

Total Records: 8760
Date Range   : 2024-01-01 00:00:00 to 2024-12-30 23:00:00
Synthetic Data Flag: True


## 2. Feature Engineering (Leakage-Free)

In [1]:
X, y, feature_names, df_processed = create_feature_pipeline(df_clean, target_col="Energy", drop_na=True)
print(f"Engineered Features ({len(feature_names)}): {feature_names}")
X.head()

Engineered Features (13): ['hour', 'dayofweek', 'month', 'is_weekend', 'lag_1', 'lag_2', 'lag_3', 'lag_24', 'lag_48', 'lag_168', 'rolling_mean_24', 'rolling_std_24', 'rolling_mean_168']


## 3. Chronological Train / Test Split

In [1]:
train_df, test_df = chronological_split(df_processed, split_ratio=0.8)
X_train, y_train = train_df[feature_names], train_df["Energy"]
X_test, y_test = test_df[feature_names], test_df["Energy"]

print(f"Train observations: {len(train_df)}")
print(f"Test observations : {len(test_df)}")

Train observations: 6873
Test observations : 1719


## 4. Benchmark Models: Naive Baseline vs Random Forest

In [1]:
# Naive Persistence Baseline (y_t = y_{t-1})
baseline = NaivePersistenceBaseline(lag_feature="lag_1")
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
metrics_base = calculate_metrics(y_test.to_numpy(), y_pred_base, model_name="Naive Persistence Baseline")

# Random Forest Regressor
rf_model = train_random_forest(X_train, y_train, n_estimators=100, max_depth=15, random_state=42)
y_pred_rf = rf_model.predict(X_test)
metrics_rf = calculate_metrics(y_test.to_numpy(), y_pred_rf, model_name="RandomForestRegressor")

metrics_df = pd.DataFrame([metrics_base, metrics_rf])
metrics_df

## 5. Recursive 24-Hour Forecast

In [1]:
forecast_df = generate_recursive_24h_forecast(
    model=rf_model,
    historical_df=df_clean,
    feature_names=feature_names,
    forecast_horizon=24,
    unit="MWh",
    is_synthetic=is_synthetic
)
forecast_df.head(10)